# 06b · Real sarcasm: Qwen vs the circuit vs the custom mechanism (iSarcasmEval)

**Question:** is the circuit we found really better at handling sarcasm, or does a simple shortcut (the custom
mechanism) give the same result on real tweets?

**Data:** iSarcasmEval (SemEval 2022, task 6), English test set: **1400 tweets, 200 sarcastic** (labelled by their
authors), with the kind of each sarcastic tweet (sarcasm, irony, satire, ...), and **200 pairs** of a sarcastic
tweet and the author's own non-sarcastic rephrase. From the training set, 100 sarcastic + 600 non-sarcastic tweets
(the same 1 : 6 ratio as the test set) are used only for the mean of the circuit and for the threshold.

Every tweet goes into the same prompt as dataset v2: *Someone said: "…" Is this person being sarcastic? Answer
with Yes or No.*

| system | how it decides |
|---|---|
| **Qwen** | d > threshold (threshold chosen on the training tweets for the best F1, because only 1 in 7 is sarcastic) |
| **circuit** (8147 edges) | the same, answering alone (mean ablation; the mean is taken over the training tweets) |
| **mechanism** | segmenter c of 06a → phrase / situation; twitter-RoBERTa sentiment; sarcastic = positive phrase **and** negative situation |

**Runtime:** L4 ≈ 45–60 min (everything is cached: if Colab stops, rerun and it continues).
Needs from Drive: the circuit (02d), the sarcasm direction (03a) and the segments of 06a.

## Setup

In [ ]:
import os, sys
REPO = "https://github.com/vasilis-pantzos/sarcasm-proposal.git"
ROOT = "/content/sarcasm-proposal"
if not os.path.exists(ROOT):
    !git clone -q {REPO} {ROOT}
else:
    !git -C {ROOT} pull -q
os.chdir(ROOT)
sys.path.insert(0, ROOT)
%pip install -q -U accelerate bitsandbytes
ISARC = "/content/iSarcasmEval"
if not os.path.exists(ISARC):
    !git clone -q https://github.com/iabufarha/iSarcasmEval.git {ISARC}

In [ ]:
import os, json, time, numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display
from src.model import load_model, model_info, DEV_MODEL, MAIN_MODEL
from src.prompts import data_tag
from src.eap import build_graph
from src.standalone import ResampledReference, StandaloneEvaluator, standalone_d
from src.pathpatch import mean_last, ablate_d
from src.implicit import SARCASM_Q, IMPLICIT_Q, project_out_d
from src.baseline import cohen_kappa
from src.segment import load_isarcasm, CATEGORIES
from src.isarcasm import (tweet_ids, tweet_prompt, twitter_sentiment, mechanism, metrics, best_threshold,
                          pick_pair, clean)
from src.utils import mount_drive, results_dir, cached, set_seed

# ---------------- settings ----------------
MODEL = MAIN_MODEL
DATA = "v2"                              # the dataset the circuit was found on
CIRCUIT = "v3b/acdc_mask_tau0.0.npy"     # the final circuit (02d)
DIRECTION = "sarcasm_direction_L19.npy"  # from 03a
PROJECT_LAYERS = [19, 20, 21]
KEY_LAYER, KEY_HEADS = 18, [15, 18, 4, 13, 16]
N_RANDOM = 3                             # random controls in the causal test
N_TRAIN_SARC, N_TRAIN_NOT = 100, 600     # training tweets (mean of the circuit + threshold)
SEED = 0
# ------------------------------------------
TAG = MODEL.split("/")[-1] + data_tag(DATA)
SUB = f"06b_isarcasm/{TAG}"
mount_drive()
print("results go to:", results_dir(SUB))
set_seed(SEED)

## 1 · Data and the segments of 06a

In [ ]:
TEST, PAIRS, TRAIN = load_isarcasm(ISARC)
TEST["text"] = TEST.text.astype(str)
y = TEST.sarcastic.values.astype(bool)
TRAIN = TRAIN.dropna(subset=["tweet"])
TR = pd.concat([TRAIN[TRAIN.sarcastic == 1].sample(N_TRAIN_SARC, random_state=SEED),
                TRAIN[TRAIN.sarcastic == 0].sample(N_TRAIN_NOT, random_state=SEED)]).reset_index(drop=True)
y_tr = TR.sarcastic.values.astype(bool)
# the sarcastic and the non-sarcastic text of every pair
PAIRS["sarcastic"] = [r.text_0 if r.sarcastic_id == 0 else r.text_1 for r in PAIRS.itertuples()]
PAIRS["rephrase"] = [r.text_1 if r.sarcastic_id == 0 else r.text_0 for r in PAIRS.itertuples()]
print(f"test: {len(TEST)} tweets ({y.sum()} sarcastic) | pairs: {len(PAIRS)} | training tweets used: {len(TR)}")

seg_dir = "06a_segmenter"
load = lambda name: torch.load(os.path.join(results_dir(seg_dir), name + ".pt"), weights_only=False)
def segments(df):
    return pd.DataFrame({"text": df.text.values, "phrase": df.c_phrase.values, "situation": df.c_situation.values})
SEG = segments(load("segments_test"))
S0, S1 = segments(load("segments_pairs_text0")), segments(load("segments_pairs_text1"))
assert (SEG.text.values == TEST.text.values).all(), "the segments of 06a do not match the test set"
pick = lambda a, b, sid: pd.DataFrame([b.iloc[i] if s == 1 else a.iloc[i] for i, s in enumerate(sid)]).reset_index(drop=True)
SEG_PS = pick(S0, S1, PAIRS.sarcastic_id.values)            # segments of the sarcastic text of each pair
SEG_PR = pick(S1, S0, PAIRS.sarcastic_id.values)            # ... of the rephrase
print("segmenter c found both parts in", f"{(SEG.phrase.notna() & SEG.situation.notna()).mean():.2f}", "of the test tweets")

## 2 · The custom mechanism

twitter-RoBERTa (trained on tweets) gives p(negative), p(neutral), p(positive) for every phrase, situation and
whole tweet. Positive phrase = p(positive) > p(negative); negative situation = p(negative) > p(positive).

In [ ]:
texts = set()
for s in (SEG, SEG_PS, SEG_PR):
    texts |= set(s.text) | set(s.phrase.dropna()) | set(s.situation.dropna())
texts = sorted(texts)
def run_sent():
    p = twitter_sentiment(texts)
    return {t: (a, b, c) for t, a, b, c in zip(texts, p["neg"], p["neu"], p["pos"])}
SENT = cached("sentiment", run_sent, sub=SUB)
torch.cuda.empty_cache()
MECH, MECH_PS, MECH_PR = mechanism(SEG, SENT), mechanism(SEG_PS, SENT), mechanism(SEG_PR, SENT)
print("mechanism says sarcastic:", int(MECH.sarcastic.sum()), "of", len(MECH), "tweets")
print(pd.crosstab(pd.Series(y, name="sarcastic (gold)"),
                  pd.Series(np.select([~MECH.found, MECH.sarcastic], ["parts not found", "contrast found"], "no contrast"),
                            name="mechanism")))

## 3 · Qwen and the circuit on every tweet

In [ ]:
t0 = time.time()
model, tok = load_model(MODEL)
G = build_graph(model)
print(MODEL, model_info(model), f"loaded in {(time.time() - t0) / 60:.1f} min")
path = os.path.join(results_dir(""), f"02_acdc/{TAG}/{CIRCUIT}")
assert os.path.exists(path), f"circuit not found: {path}"
MASK = np.load(path)
print(f"circuit: {int(MASK.sum())} edges")

IDS_TR, P0, S0_ = tweet_ids(tok, TR.tweet)
IDS_TE, P1, S1_ = tweet_ids(tok, TEST.text)
IDS_PS, P2, S2_ = tweet_ids(tok, PAIRS.sarcastic)
IDS_PR, P3, S3_ = tweet_ids(tok, PAIRS.rephrase)
PREFIX, SUFFIX = min(P0, P1, P2, P3), min(S0_, S1_, S2_, S3_)
print("template tokens before / after the tweet:", PREFIX, SUFFIX,
      "| prompt length: median", int(np.median([len(x) for x in IDS_TE])), "max", max(len(x) for x in IDS_TE))
print(tweet_prompt(TEST.text.iloc[0]))

t0 = time.time()
REF = ResampledReference(model, IDS_TR, G, prefix=PREFIX, suffix=SUFFIX)
print(f"mean reference: {len(IDS_TR)} training tweets, middle resampled to {REF.middle} positions | {time.time() - t0:.0f} s")

# sanity: all edges = the full model
T0 = pd.Series([len(x) for x in IDS_TE]).mode()[0]
part = [x for x in IDS_TE if len(x) == T0][:8]
ev = StandaloneEvaluator(model, tok, part, REF, G)
gap = np.abs(ev.d(G.valid) - ev.d_clean).max()
print(f"all edges: max |d - d_full| = {gap:.5f}")
assert gap < 1e-3, "sanity check failed: send the output to Claude"
del ev; torch.cuda.empty_cache()
print("PASS")

In [ ]:
t0 = time.time()
run = lambda ids: standalone_d(model, tok, ids, {"circuit": MASK}, REF, G, batch=16, progress=False)
D_TR = cached("d_train", lambda: run(IDS_TR), sub=SUB)
D_TE = cached("d_test", lambda: run(IDS_TE), sub=SUB)
D_PS = cached("d_pairs_sarcastic", lambda: run(IDS_PS), sub=SUB)
D_PR = cached("d_pairs_rephrase", lambda: run(IDS_PR), sub=SUB)
print(f"{(time.time() - t0) / 60:.1f} min")

THR = {k: best_threshold(D_TR[k], y_tr) for k in ["full", "circuit"]}
print("thresholds (best F1 on the training tweets):", {k: round(v, 2) for k, v in THR.items()})
print("for reference, Qwen with d > 0 on the test set:", {k: round(v, 3) for k, v in metrics(D_TE["full"] > 0, y).items()})
TEST["d Qwen"], TEST["d circuit"] = D_TE["full"], D_TE["circuit"]
YES = {"Qwen": D_TE["full"] > THR["full"], "circuit": D_TE["circuit"] > THR["circuit"],
       "mechanism": MECH.sarcastic.values}

## 4 · Main table

F1, precision and recall of the **sarcastic** class (the usual measure in iSarcasmEval; the best systems of the
competition reached F1 ≈ 0.6), accuracy, the share of tweets answered "sarcastic", and Cohen's κ with Qwen.

In [ ]:
MAIN = pd.DataFrame({k: {**metrics(v, y), "κ with Qwen": cohen_kappa(v, YES["Qwen"])} for k, v in YES.items()}).T
display(MAIN.round(3))

## 5 · Where do they differ?

**(a)** On the sarcastic tweets: which ones each system finds. **(b)** Recall per kind of sarcasm. **(c)** The
key split: tweets where the mechanism finds the contrast "positive phrase + negative situation" and all the others.
If Qwen is only a better version of the shortcut, its advantage should be inside the contrast group; if it handles
sarcasm the shortcut cannot see, it should also find sarcastic tweets outside it.

In [ ]:
q, m, c = YES["Qwen"], YES["mechanism"], YES["circuit"]
print("(a) sarcastic tweets found (of", y.sum(), ")")
display(pd.crosstab(pd.Series(np.where(q[y], "Qwen: yes", "Qwen: no"), name=""),
                    pd.Series(np.where(m[y], "mechanism: yes", "mechanism: no"), name="")))

rows = []
for cat in CATEGORIES:
    k = (TEST[cat] == 1).values & y
    if k.sum():
        rows.append({"kind": cat, "n": int(k.sum()), **{s: float(v[k].mean()) for s, v in YES.items()}})
CAT = pd.DataFrame(rows).set_index("kind")
print("(b) recall per kind of sarcasm")
display(CAT.round(2))

CONTRAST = MECH.sarcastic.values
rows = []
for name, g in [("contrast found", CONTRAST), ("no contrast", ~CONTRAST)]:
    for s in ["Qwen", "circuit"]:
        rows.append({"group": name, "system": s, "tweets": int(g.sum()), "sarcastic": int(y[g].sum()),
                     "recall": float(YES[s][g & y].mean()) if (g & y).any() else np.nan,
                     "said Yes to non-sarcastic": float(YES[s][g & ~y].mean()) if (g & ~y).any() else np.nan})
SPLIT = pd.DataFrame(rows).set_index(["group", "system"])
print("(c) inside / outside the contrast group")
display(SPLIT.round(3))
print(f"share of all sarcastic tweets that are in the contrast group: {CONTRAST[y].mean():.2f}")

## 6 · Pairs: detection and interpretation

**Detection** (task C of iSarcasmEval): which of the two texts is the sarcastic one? Qwen and the circuit pick the
text with the higher d, the mechanism the text with the higher score p(positive phrase) · p(negative situation)
(a tie counts as half).

**Interpretation:** the implicit question of phase 5, *"Is this person actually happy about it?"*, on the sarcastic
tweet and on the author's rephrase. The rephrase says what the author meant, so a system that understands the
sarcasm gives the **same** answer to both. The mechanism answers with the sentiment of the situation (of the whole
text if no parts were found); the **literal** reader answers with the sentiment of the whole text.

In [ ]:
DET = {"Qwen": pick_pair(D_PR["full"], D_PS["full"]), "circuit": pick_pair(D_PR["circuit"], D_PS["circuit"]),
       "mechanism": pick_pair(MECH_PR.score, MECH_PS.score)}
DET = pd.Series({k: float(v.mean()) for k, v in DET.items()}, name="detection accuracy")

IMP_TR, a0, b0 = tweet_ids(tok, TR.tweet, IMPLICIT_Q)
IMP_PS, a1, b1 = tweet_ids(tok, PAIRS.sarcastic, IMPLICIT_Q)
IMP_PR, a2, b2 = tweet_ids(tok, PAIRS.rephrase, IMPLICIT_Q)
t0 = time.time()
REF_IMP = ResampledReference(model, IMP_TR, G, prefix=min(a0, a1, a2), suffix=min(b0, b1, b2))
runi = lambda ids: standalone_d(model, tok, ids, {"circuit": MASK}, REF_IMP, G, batch=16, progress=False)
DI_PS = cached("d_implicit_pairs_sarcastic", lambda: runi(IMP_PS), sub=SUB)
DI_PR = cached("d_implicit_pairs_rephrase", lambda: runi(IMP_PR), sub=SUB)
print(f"{(time.time() - t0) / 60:.1f} min")

HAPPY = {"Qwen": (DI_PS["full"] > 0, DI_PR["full"] > 0), "circuit": (DI_PS["circuit"] > 0, DI_PR["circuit"] > 0),
         "mechanism": (MECH_PS.happy.values, MECH_PR.happy.values),
         "literal": (MECH_PS["literal happy"].values, MECH_PR["literal happy"].values)}
PAIR = pd.DataFrame({k: {"same answer to both": float((a == b).mean()), "'not happy' on the sarcastic tweet": float((~a).mean()),
                         "'not happy' on the rephrase": float((~b).mean())} for k, (a, b) in HAPPY.items()}).T
PAIR["detection accuracy"] = DET
display(PAIR.round(3))

## 7 · Causal test on Qwen: does it use the mechanism we found on real tweets?

As in phase 5: remove the sarcasm direction (layers 19–21) or the layer-18 writers (m18 + 5 key heads), against
random controls. Same threshold as before. If the found mechanism is what Qwen uses for real contrast sarcasm, the
recall should drop **mostly inside the contrast group**.

In [ ]:
pdir = os.path.join(results_dir(""), f"03a_direction/{TAG}/{DIRECTION}")
assert os.path.exists(pdir), f"direction not found: {pdir}"
U = np.load(pdir).astype(np.float32)
rng = np.random.default_rng(0)
RAND_DIRS = [rng.standard_normal(U.shape).astype(np.float32) for _ in range(N_RANDOM)]
H = model_info(model)["n_heads"]
RAND_HEADS = [sorted(rng.choice([h for h in range(H) if h not in KEY_HEADS], 5, replace=False).tolist())
              for _ in range(N_RANDOM)]

def ablations():
    z, mm = mean_last(model, IDS_TR, KEY_LAYER)
    out = {"direction (L19-21)": project_out_d(model, tok, IDS_TE, PROJECT_LAYERS, U),
           "writers (m18 + 5 heads)": ablate_d(model, tok, IDS_TE, KEY_LAYER, KEY_HEADS, True, z, mm)}
    for i, v in enumerate(RAND_DIRS):
        out[f"random direction #{i}"] = project_out_d(model, tok, IDS_TE, PROJECT_LAYERS, v)
    for i, hs in enumerate(RAND_HEADS):
        out[f"random 5 heads #{i}"] = ablate_d(model, tok, IDS_TE, KEY_LAYER, hs, False, z, mm)
    return out

t0 = time.time()
ABL = {"none": D_TE["full"], **cached("ablations", ablations, sub=SUB)}
print(f"{(time.time() - t0) / 60:.1f} min")
rows = []
for k, d in ABL.items():
    yes = d > THR["full"]
    rows.append({"ablation": k, "F1": metrics(yes, y)["F1"],
                 "recall: contrast found": float(yes[CONTRAST & y].mean()),
                 "recall: no contrast": float(yes[~CONTRAST & y].mean()),
                 "said Yes to non-sarcastic": float(yes[~y].mean()),
                 "mean d (sarcastic)": float(d[y].mean())})
CAUS = pd.DataFrame(rows).set_index("ablation")
display(CAUS.round(3))

## 8 · Figure

In [ ]:
plt.rcParams.update({"font.size": 12})
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
x = np.arange(3)
for j, s in enumerate(YES):
    ax.bar(x + (j - 1) * 0.27, MAIN.loc[s, ["F1", "precision", "recall"]], 0.27, label=s)
ax.set_xticks(x); ax.set_xticklabels(["F1", "precision", "recall"]); ax.set_ylim(0, 1); ax.legend()
ax = axes[1]
keys = ["none", "direction (L19-21)", "writers (m18 + 5 heads)", "random direction #0", "random 5 heads #0"]
x = np.arange(len(keys))
ax.bar(x - 0.2, CAUS.loc[keys, "recall: contrast found"], 0.4, label="contrast found")
ax.bar(x + 0.2, CAUS.loc[keys, "recall: no contrast"], 0.4, label="no contrast")
ax.set_xticks(x); ax.set_xticklabels(["none", "direction", "writers", "random dir.", "random heads"])
ax.set_ylabel("recall (Qwen)"); ax.set_ylim(0, 1); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "isarcasm.png"), dpi=200); plt.show()

## 9 · Examples where Qwen and the mechanism disagree (sarcastic tweets)

In [ ]:
pd.set_option("display.max_colwidth", 90)
EX = pd.DataFrame({"tweet": TEST.text, "phrase": SEG.phrase, "situation": SEG.situation,
                   "d Qwen": TEST["d Qwen"].round(2), "Qwen": YES["Qwen"], "mechanism": YES["mechanism"]})
print("Qwen finds it, the mechanism does not:")
display(EX[y & q & ~m].head(8))
print("the mechanism finds it, Qwen does not:")
display(EX[y & ~q & m].head(8))
print("the mechanism says sarcastic, but the tweet is not:")
display(EX[~y & m].head(5))

## Save

In [ ]:
d = results_dir(SUB)
out = TEST[["text", "sarcastic", *CATEGORIES, "d Qwen", "d circuit"]].copy()
out["phrase"], out["situation"] = SEG.phrase, SEG.situation
for k, v in YES.items():
    out[f"yes {k}"] = v
out["contrast found"] = CONTRAST
out.to_csv(os.path.join(d, "per_tweet.csv"), index=False)
for name, t in [("main", MAIN), ("per_kind", CAT), ("contrast_split", SPLIT), ("pairs", PAIR), ("causal", CAUS)]:
    t.to_csv(os.path.join(d, f"{name}.csv"))
json.dump(THR, open(os.path.join(d, "thresholds.json"), "w"))
print("saved to", d)